In [4]:
# ===== Noise control: real cells vs random curves, frame split vs block split =====
# Paste this whole cell into a notebook and run it. Change the settings below.

# ---------------- settings ----------------
REPO_ROOT = "D:/SI_FILES/Research/UIUC Neroscience Research/Bio-Analysis"   # <-- change to where the repo is on your computer
FOV       = ["K215_A_D1"]   # one or more tokens; a prefix like "K215_A" picks all matching; [] = all FOVs
BLOCKS    = 10              # number of contiguous time blocks for the block split
N_NOISE   = 20              # number of random curves
SEED      = 0               # random seed for curves and frame shuffling
OUT_CSV   = None            # e.g. "result.csv" to save, None to skip
# ------------------------------------------

import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold, GroupKFold

warnings.filterwarnings("ignore")
REPO_ROOT = Path(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# src_predict reads "../config/.env.path" relative to the working folder, so run from script/
os.chdir(REPO_ROOT / "script")
from src_predict import Preprocessor

CFG_FOLDER = REPO_ROOT / "cache" / "rfecv_init_vote" / "cont"
DATA_FOLDER = REPO_ROOT / "data" / "initialization"


def pick_tokens(requested):
    tokens = sorted(Path(f).name[:-len("_P.json")] for f in glob.glob(str(CFG_FOLDER / "*_P.json")))
    if isinstance(requested, str):
        requested = [requested]
    if not requested:
        return tokens
    picked = []
    for r in requested:
        hits = [t for t in tokens if t == r] or [t for t in tokens if t.startswith(r)]
        if not hits:
            print(f"[skip] no FOV matches '{r}'. Available: {', '.join(tokens)}")
        picked += [h for h in hits if h not in picked]
    return picked


def load_fov(token):
    """Same preprocessing config that rfecv_select used; returns time-sorted original frames."""
    with open(CFG_FOLDER / f"{token}_P.json") as fp:
        cfg = json.load(fp)
    cfg["beh_path"] = DATA_FOLDER / f"{token}_B.csv"
    cfg["sig_path"] = DATA_FOLDER / f"{token}_S.csv"
    total = Preprocessor(**cfg, log_level="ERROR").get_origin_total(shard="all").sort_index()
    return total.iloc[:, :-1].values, total.iloc[:, -1].values.astype(int)


def cv_score(x, y, split, n_blocks, seed):
    """'frame': shuffled 10-fold CV. 'block': contiguous time blocks, each tested once."""
    if split == "frame":
        folds = KFold(n_splits=10, shuffle=True, random_state=seed).split(x)
    else:
        groups = np.arange(len(y)) * n_blocks // len(y)
        folds = GroupKFold(n_splits=n_blocks).split(x, y, groups)
    pred = np.zeros_like(y)
    for tr, te in folds:
        model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=10,
                                   num_leaves=31, verbose=-1, random_state=42)
        model.fit(x[tr], y[tr])
        pred[te] = model.predict(x[te])
    return balanced_accuracy_score(y, pred)


rows = []
for token in pick_tokens(FOV):
    try:
        x, y = load_fov(token)
    except Exception as e:
        print(f"[skip] {token}: {e}")
        continue
    rng = np.random.default_rng(SEED)
    curves = np.cumsum(rng.standard_normal((len(y), N_NOISE)), axis=0)   # random-walk curves
    row = {
        "fov": token,
        "n_cells": x.shape[1],
        "real_frame": cv_score(x, y, "frame", BLOCKS, SEED),
        "noise_frame": cv_score(curves, y, "frame", BLOCKS, SEED),
        "real_block": cv_score(x, y, "block", BLOCKS, SEED),
        "noise_block": cv_score(curves, y, "block", BLOCKS, SEED),
    }
    rows.append(row)
    print(f"{token}: real_frame={row['real_frame']:.3f}  noise_frame={row['noise_frame']:.3f}  "
          f"real_block={row['real_block']:.3f}  noise_block={row['noise_block']:.3f}")

df = pd.DataFrame(rows)
if len(df) > 1:
    print("\nMean over selected FOVs (balanced accuracy, chance = 0.5):")
    print(df[["real_frame", "noise_frame", "real_block", "noise_block"]].mean().round(3).to_string())
if OUT_CSV and len(df):
    df.to_csv(OUT_CSV, index=False)
    print(f"\nSaved to {OUT_CSV}")
df

  File "D:\SW_Anaconda\envs\regular\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "D:\SW_Anaconda\envs\regular\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\SW_Anaconda\envs\regular\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "D:\SW_Anaconda\envs\regular\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


K215_A_D1: real_frame=0.895  noise_frame=0.927  real_block=0.590  noise_block=0.632


,fov,n_cells,real_frame,noise_frame,real_block,noise_block
0,K215_A_D1,23,0.894866,0.927079,0.589651,0.632498
